## Content

- [1. Check environment and library version](#1-checking-environment-and-library-version)

- [2. Extract Ticker from Target Sector](#2-extract-ticker-from-it-sector)


### 1. Checking environment and library version

In [1]:
### Check common ML libray and its version
## >>> It is common to experience conflict with pyarrow, tensorflow and mlflow
## >>> There is a suspection of is a binary incompatibility between PyArrow and protobuf / abseil-cpp libraries inside that conda environment.

import numpy as np; print("NumPy version:", np.__version__)
import pandas as pd; print("Panda version:", pd.__version__)
# import xgboost as xgb; print("XGBoost version:", xgb.__version__)
import sklearn; print("scikit-learn version:", sklearn.__version__)

# import pyarrow; print("pyarrow version:", pyarrow.__version__)
import tensorflow as tf; print("tensorflow version:", tf.__version__)
import mlflow; print("MLflow version:", mlflow.__version__)


NumPy version: 1.26.4
Panda version: 2.3.1
scikit-learn version: 1.6.1


2025-11-22 20:19:53.396041: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-22 20:19:53.414321: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-22 20:19:53.419331: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-22 20:19:53.432545: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-22 20:19:54.098593: W tensorflow/compiler/tf2

tensorflow version: 2.17.0


/home/yikai/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.1.4) and mlflow-skinny (2.22.0) are different. This may lead to unexpected behavior. Please install the same version of both packages.
  mlflow.mismatch._check_version_mismatch()


MLflow version: 2.22.0


### 2. Extract Ticker from IT Sector

In [2]:
## >> 1.1 System Setup and Load Libraries
# import time 
# start_time = time.time()

FIG_PATH = 'plots'
HTML_PATH = '../html/test16'  ## <- change to proper name

import os # sys
os.makedirs("plots", exist_ok=True)
os.makedirs(f"{HTML_PATH}", exist_ok=True)

import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
df = pd.read_csv("../data/raw/prices-split-adjusted.csv", parse_dates=["date"])
df['symbol'].value_counts()

symbol
KSU     1762
NOC     1762
ORCL    1762
OMC     1762
OKE     1762
        ... 
KHC      378
HPE      304
CSRA     284
WLTW     251
FTV      126
Name: count, Length: 501, dtype: int64

In [4]:
df_filtered = df.groupby('symbol').filter(lambda x: len(x) > 1500) # make all the data is in the same length
print(f"Total Ticker :{len(df['symbol'].value_counts())}")
print(f"Ticker with > 1500 records :{len(df_filtered['symbol'].value_counts())}")
df_filtered


Total Ticker :501
Ticker with > 1500 records :470


,date,symbol,open,close,low,high,volume
251,2010-01-04,A,22.453504,22.389128,22.267525,22.625180,3815500.0
252,2010-01-04,AAL,4.840000,4.770000,4.660000,4.940000,9837300.0
253,2010-01-04,AAP,40.700001,40.380001,40.360001,41.040001,1701700.0
254,2010-01-04,AAPL,30.490000,30.572857,30.340000,30.642857,123432400.0
255,2010-01-04,ABC,26.290001,26.629999,26.139999,26.690001,2455900.0
...,...,...,...,...,...,...,...
851257,2016-12-30,YHOO,38.720001,38.669998,38.430000,39.000000,6431600.0
851258,2016-12-30,YUM,63.930000,63.330002,63.160000,63.939999,1887100.0
851259,2016-12-30,ZBH,103.309998,103.199997,102.849998,103.930000,973800.0
851260,2016-12-30,ZION,43.070000,43.040001,42.689999,43.310001,1938100.0


In [22]:
### >>  map price data to the GICs sectors

df_security = pd.read_csv("../data/raw/securities.csv")
# creates a dictionary that maps ticker to its GICS sector
tick_to_sec = dict(zip(df_security['Ticker symbol'], df_security['GICS Sector']))
tick_to_subsec = dict(zip(df_security['Ticker symbol'], df_security['GICS Sub Industry']))

## >>  map the ticker to the price data << ## 
df_price = df_filtered.copy()
df_price['Sector'] = df_price['symbol'].map(tick_to_sec)
df_price['Subsector'] = df_price['symbol'].map(tick_to_subsec)
display(df_price)

## >> check for NA nad INF before removing it
# check if df has any NA

has_nan = df_price.isna().any().any()
print(f"Contain NA: {has_nan}")  # False

# check if has any INF
numeric_cols = df_price.select_dtypes(include=[np.number])
has_inf = numeric_cols.isin([np.inf, -np.inf]).any().any()
print(f"Contain INF: {has_inf}")  # True

,date,symbol,open,close,low,high,volume,Sector,Subsector
251,2010-01-04,A,22.453504,22.389128,22.267525,22.625180,3815500.0,Health Care,Health Care Equipment
252,2010-01-04,AAL,4.840000,4.770000,4.660000,4.940000,9837300.0,Industrials,Airlines
253,2010-01-04,AAP,40.700001,40.380001,40.360001,41.040001,1701700.0,Consumer Discretionary,Automotive Retail
254,2010-01-04,AAPL,30.490000,30.572857,30.340000,30.642857,123432400.0,Information Technology,Computer Hardware
255,2010-01-04,ABC,26.290001,26.629999,26.139999,26.690001,2455900.0,Health Care,Health Care Distributors
...,...,...,...,...,...,...,...,...,...
851257,2016-12-30,YHOO,38.720001,38.669998,38.430000,39.000000,6431600.0,Information Technology,Internet Software & Services
851258,2016-12-30,YUM,63.930000,63.330002,63.160000,63.939999,1887100.0,Consumer Discretionary,Restaurants
851259,2016-12-30,ZBH,103.309998,103.199997,102.849998,103.930000,973800.0,Health Care,Health Care Equipment
851260,2016-12-30,ZION,43.070000,43.040001,42.689999,43.310001,1938100.0,Financials,Regional Banks


Contain NA: False
Contain INF: False


### Fundamentals.csv

In [6]:
## load fundamentals
df_fundamentals = pd.read_csv("../data/raw/fundamentals.csv")

## >> only use ticker from df_price

# Get tickers from df_price (470 tickers)
valid_tickers = set(df_price['symbol'].unique())

# Get tickers from df_fundamentals (original set)
original_tickers = set(df_fundamentals['Ticker Symbol'].unique())

# Find which tickers to remove
tickers_to_remove = original_tickers - valid_tickers
print(f"Removing {len(tickers_to_remove)} tickers: {tickers_to_remove}")

# Filter df_fundamentals
df_fundamentals = df_fundamentals[df_fundamentals['Ticker Symbol'].isin(valid_tickers)]
df_fundamentals_sector = df_fundamentals.copy()
df_fundamentals_sector['Sector'] = df_fundamentals_sector["Ticker Symbol"].map(tick_to_sec)
df_fundamentals_sector['Subsector'] = df_fundamentals_sector["Ticker Symbol"].map(tick_to_subsec)

Removing 28 tickers: {'SYF', 'CFG', 'QRVO', 'HPE', 'PYPL', 'DLPH', 'PSX', 'WRK', 'NAVI', 'NWSA', 'COTY', 'NLSN', 'KORS', 'CSRA', 'KMI', 'ABBV', 'HCA', 'WLTW', 'NWS', 'FBHS', 'TDG', 'MPC', 'XYL', 'ALLE', 'UA', 'FB', 'ZTS', 'TRIP'}


In [7]:
print(f"Total Ticker :{len(df['symbol'].value_counts())}")
print(f"Ticker with > 1500 records :{len(df_filtered['symbol'].value_counts())}")
print(f"Total number of Tikcer in funadmentals with >1500 records: {len(df_fundamentals_sector['Ticker Symbol'].unique())}")



Total Ticker :501
Ticker with > 1500 records :470
Total number of Tikcer in funadmentals with >1500 records: 420


In [28]:
tick_to_security = dict(zip(df_security['Ticker symbol'], df_security['Security']))

In [30]:
df_ticker = pd.DataFrame()
df_ticker['Symbol'] = df_fundamentals_sector['Ticker Symbol'].unique()
df_ticker['Security'] = df_ticker['Symbol'].map(tick_to_security)
df_ticker['Sector'] = df_ticker['Symbol'].map(tick_to_sec)
df_ticker['Subcector'] = df_ticker['Symbol'].map(tick_to_subsec)
df_ticker
df_ticker.to_csv('ticker.csv', index=False)

In [31]:
len(df_ticker)

420

In [45]:
print(df_fundamentals_sector.columns)
len(df_fundamentals_sector.columns)

Index(['Unnamed: 0', 'Ticker Symbol', 'Period Ending', 'Accounts Payable',
       'Accounts Receivable', 'Add'l income/expense items', 'After Tax ROE',
       'Capital Expenditures', 'Capital Surplus', 'Cash Ratio',
       'Cash and Cash Equivalents', 'Changes in Inventories', 'Common Stocks',
       'Cost of Revenue', 'Current Ratio', 'Deferred Asset Charges',
       'Deferred Liability Charges', 'Depreciation',
       'Earnings Before Interest and Tax', 'Earnings Before Tax',
       'Effect of Exchange Rate',
       'Equity Earnings/Loss Unconsolidated Subsidiary', 'Fixed Assets',
       'Goodwill', 'Gross Margin', 'Gross Profit', 'Income Tax',
       'Intangible Assets', 'Interest Expense', 'Inventory', 'Investments',
       'Liabilities', 'Long-Term Debt', 'Long-Term Investments',
       'Minority Interest', 'Misc. Stocks', 'Net Borrowings', 'Net Cash Flow',
       'Net Cash Flow-Operating', 'Net Cash Flows-Financing',
       'Net Cash Flows-Investing', 'Net Income', 'Net Income Ad

81

In [46]:
df_fundamentals_sector['Sector'].value_counts()

Sector
Consumer Discretionary         292
Industrials                    228
Information Technology         222
Financials                     196
Health Care                    182
Consumer Staples               128
Energy                         112
Real Estate                    108
Utilities                       96
Materials                       92
Telecommunications Services     20
Name: count, dtype: int64

In [47]:
df_fundamentals_sector['Subsector'].value_counts().head(5)

Subsector
Industrial Conglomerates              72
Internet Software & Services          64
Oil & Gas Exploration & Production    64
Health Care Equipment                 56
REITs                                 56
Name: count, dtype: int64

### Principal Component Analysis

In [48]:
##### >>> Only Select Information Technology Sector <<< ##### 

df_fundamentals_sector = df_fundamentals_sector[df_fundamentals_sector['Sector']=='Information Technology']
df_fundamentals_sector.head(10)


,Unnamed: 0,Ticker Symbol,Period Ending,Accounts Payable,Accounts Receivable,Add'l income/expense items,After Tax ROE,Capital Expenditures,Capital Surplus,Cash Ratio,...,Total Equity,Total Liabilities,Total Liabilities & Equity,Total Revenue,Treasury Stock,For Year,Earnings Per Share,Estimated Shares Outstanding,Sector,Subsector
8,8,AAPL,2013-09-28,3.622300e+10,-1.949000e+09,1.156000e+09,30.0,-8.165000e+09,0.000000e+00,93.0,...,1.235490e+11,8.345100e+10,2.070000e+11,1.709100e+11,0.000000e+00,2013.0,40.03,9.252311e+08,Information Technology,Computer Hardware
9,9,AAPL,2014-09-27,4.864900e+10,-6.452000e+09,9.800000e+08,35.0,-9.571000e+09,0.000000e+00,40.0,...,1.115470e+11,1.202920e+11,2.318390e+11,1.827950e+11,0.000000e+00,2014.0,6.49,6.087827e+09,Information Technology,Computer Hardware
10,10,AAPL,2015-09-26,6.067100e+10,-3.124000e+09,1.285000e+09,45.0,-1.124700e+10,0.000000e+00,52.0,...,1.193550e+11,1.709900e+11,2.903450e+11,2.337150e+11,0.000000e+00,2015.0,9.28,5.753664e+09,Information Technology,Computer Hardware
11,11,AAPL,2016-09-24,5.932100e+10,1.044000e+09,1.348000e+09,36.0,-1.273400e+10,0.000000e+00,85.0,...,1.282490e+11,1.934370e+11,3.216860e+11,2.156390e+11,0.000000e+00,2016.0,8.35,5.471497e+09,Information Technology,Computer Hardware
24,24,ADBE,2013-11-29,7.292570e+08,3.364900e+07,9.260000e+05,4.0,-1.883580e+08,3.392696e+09,208.0,...,6.724634e+09,3.655664e+09,1.038030e+10,4.055240e+09,-3.643190e+09,2013.0,0.58,4.999741e+08,Information Technology,Application Software
25,25,ADBE,2014-11-28,7.761630e+08,7.928000e+06,8.423000e+06,4.0,-1.483320e+08,3.778495e+09,150.0,...,6.775905e+09,4.009924e+09,1.078583e+10,4.147065e+09,-3.918851e+09,2014.0,0.54,4.970278e+08,Information Technology,Application Software
26,26,ADBE,2015-11-27,7.793560e+08,-7.950200e+07,3.487000e+07,9.0,-1.849360e+08,4.184883e+09,180.0,...,7.001580e+09,4.724892e+09,1.172647e+10,4.795511e+09,-4.267715e+09,2015.0,1.26,4.996437e+08,Information Technology,Application Software
27,27,ADBE,2016-12-02,8.660160e+08,-1.604160e+08,1.197800e+07,16.0,-2.038050e+08,4.616331e+09,169.0,...,7.424835e+09,5.282279e+09,1.270711e+10,5.854430e+09,-5.132472e+09,NaN,NaN,NaN,Information Technology,Application Software
28,28,ADI,2013-11-02,3.230840e+08,1.237700e+07,8.935000e+07,14.0,-1.230740e+08,7.118790e+08,821.0,...,4.739576e+09,1.642174e+09,6.381750e+09,2.633689e+09,0.000000e+00,2013.0,2.19,3.075283e+08,Information Technology,Semiconductors
29,29,ADI,2014-11-01,4.306210e+08,-3.646000e+07,1.164500e+07,13.0,-1.779130e+08,6.430580e+08,404.0,...,4.757897e+09,2.101793e+09,6.859690e+09,2.864773e+09,0.000000e+00,2014.0,0.35,1.798057e+09,Information Technology,Semiconductors


In [49]:
print(f"Ticker: {len(df_fundamentals_sector['Ticker Symbol'].value_counts())}")
print(f"Sum of the total Ticker count: {sum(df_fundamentals_sector['Ticker Symbol'].value_counts())}")

Ticker: 56
Sum of the total Ticker count: 222


In [55]:
df_fundamentals_sector['Ticker Symbol'].unique()

array(['AAPL', 'ADBE', 'ADI', 'ADS', 'ADSK', 'AKAM', 'AMAT', 'APH',
       'ATVI', 'AVGO', 'CRM', 'CSCO', 'CTSH', 'CTXS', 'EA', 'EBAY',
       'FFIV', 'FIS', 'FISV', 'FLIR', 'FSLR', 'GLW', 'GPN', 'HPQ', 'HRS',
       'IBM', 'INTC', 'INTU', 'JNPR', 'KLAC', 'LLTC', 'LRCX', 'MA',
       'MCHP', 'MSFT', 'MU', 'NFLX', 'NTAP', 'NVDA', 'PAYX', 'QCOM',
       'RHT', 'STX', 'SWKS', 'SYMC', 'TDC', 'TEL', 'TSS', 'TXN', 'V',
       'VRSN', 'WDC', 'WU', 'XLNX', 'XRX', 'YHOO'], dtype=object)

In [51]:
Batch_1 = ['AAPL', 'ADBE', 'ADI', 'ADS', 'ADSK', 'AKAM', 'AMAT', 'APH',
       'ATVI', 'AVGO', 'CRM', 'CSCO', 'CTSH', 'CTXS', 'EA', 'EBAY',
       'FFIV', 'FIS', 'FISV', 'FLIR', 'FSLR', 'GLW', 'GPN', 'HPQ', 'HRS']
       

Batch_2 = ['IBM', 'INTC', 'INTU', 'JNPR', 'KLAC', 'LLTC', 'LRCX', 'MA',
       'MCHP', 'MSFT', 'MU', 'NFLX', 'NTAP', 'NVDA', 'PAYX', 'QCOM',
       'RHT', 'STX', 'SWKS', 'SYMC', 'TDC', 'TEL', 'TSS', 'TXN', 'V',
       'VRSN', 'WDC', 'WU', 'XLNX', 'XRX', 'YHOO']

In [52]:
subsector_counts = df_fundamentals_sector[df_fundamentals_sector['Sector'] == 'Information Technology']['Subsector'].value_counts().reset_index()
subsector_counts.columns = ['Subsector', 'Count']
print(subsector_counts)

                                Subsector  Count
0            Internet Software & Services     64
1                          Semiconductors     46
2                    Application Software     16
3                 Semiconductor Equipment     12
4                    Networking Equipment     12
5          IT Consulting & Other Services     12
6                       Computer Hardware      8
7   Data Processing & Outsourced Services      8
8                   Electronic Components      8
9             Home Entertainment Software      8
10                       Systems Software      8
11         Computer Storage & Peripherals      8
12     Electronic Equipment & Instruments      4
13           Telecommunications Equipment      4
14      Electronic Manufacturing Services      4
